# Import the rsm3d module and set the data input/oupt directories

In [20]:
from rsm3d.rsm3d import RSMBuilder  # <-- 4-circle xrayutilities builder
from rsm3d.data_viz import RSMNapariViewer
spec_file = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/setup_6oct23'
tiff_dir  = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/data_6oct23_tiff'
tiff_output = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/data_6oct23_tiff_cleaned'
scan_list = (21,)     # any list/tuple of scan numbers
out_vtr   = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/rsm_hkl.vtr'    # output file

In [2]:
from rsm3d.utilis import remove_dead_pixels_in_dir
mask = remove_dead_pixels_in_dir(tiff_dir, tiff_output)

# Call the RSMBuilder and compute the Q-sample and HKL for the listed frames

In [21]:
builder = RSMBuilder(
    spec_file, tiff_dir,
    selected_scans=scan_list,
    ub_includes_2pi=True,        # set False if your UB is "no-2π"
    center_is_one_based=False,)   # True if SPEC xcenter/ycenter are 1-based )
Q_samp, hkl, intensity = builder.compute_full()


Initialized QConversion area with:
  Sample Axis: ['x+', 'y+', 'z-']
  Detector Axis: ['x+']
  Beam Direction: (0, 1, 0)
  Wavelength: 1.080943 Å
  Distance: 0.781050 m
  Pixel Width: 0.000075 m


# Mapping the intensity with the HKL/Q_samp for 3D visualization

In [22]:
# Optional cropping
# builder.crop_by_positions(y_bound=(220, 510), x_bound=(380, 610))
# now builder.hkl, builder.Q_samp, builder.intensity are cropped

grid, (xax, yax, zax) = builder.regrid_xu(
   space="hkl",
   grid_shape=(200, None, None),
   fuzzy=True,
   normalize="mean",
   stream=True
)

# Call the napari for the 3D visualization of the RSM map

In [23]:
viz = RSMNapariViewer(
    grid, (xax, yax, zax),
    space="hkl",               # or "q"
    name="RSM",
    log_view=True,
    contrast_percentiles=(1, 99.8),
    cmap="inferno",
    rendering="attenuated_mip",  # or "mip", "translucent"
)
# launch returns the raw napari.Viewer
viewer = viz.launch()

In [24]:
# Optional cropping
builder.crop_by_positions(y_bound=(260, 510), x_bound=(380, 610))
# now builder.hkl, builder.Q_samp, builder.intensity are cropped

grid, (xax, yax, zax) = builder.regrid_xu(
   space="hkl",
   grid_shape=(200, None, None),
   fuzzy=True,
   normalize="mean",
   stream=True
)
viz = RSMNapariViewer(
    grid, (xax, yax, zax-0.06558),
    space="hkl",               # or "q"
    name="RSM",
    log_view=True,
    contrast_percentiles=(1, 99.8),
    cmap="inferno",
    rendering="attenuated_mip",  # or "mip", "translucent"
)
# launch returns the raw napari.Viewer
viewer = viz.launch()

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
File ~/pyprojects/pyisr/.pixi/envs/default/lib/python3.13/site-packages/napari/_qt/threads/status_checker.py:126, in StatusChecker.calculate_status(self=<napari._qt.threads.status_checker.StatusChecker(0x3c10f85c0, name = "StatusChecker")>)
    122     return
    124 try:
    125     # Calculate the status change from cursor's movement
--> 126     res = viewer._calc_status_from_cursor()
        viewer = Viewer(camera=Camera(center=(np.float64(-0.03413189464986324), np.float64(3.9337782859802246), np.float64(3.9564608335494995)), zoom=np.float64(2061.8275462513866), angles=(np.float64(4.096910883926571), np.float64(47.50686506572259), np.float64(145.38785768718762)), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.

In [9]:
from rsm3d.data_io import write_rsm_volume_to_vtr, write_rsm_volume_to_vtk
rsm = grid
edges = (xax, yax, zax)
filename = out_vtr
write_rsm_volume_to_vtk(rsm, edges, filename.replace('.vtr', '.vtk'))
write_rsm_volume_to_vtr(rsm, edges, filename, binary=False, compress=True)
